In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv("UNSW_NB15_training-set.csv" ) 


test = pd.read_csv("UNSW_NB15_testing-set.csv" )


In [3]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [4]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [5]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [6]:
y = train.pop("attack_cat").values
X = train.values

In [7]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [8]:
from sklearn.decomposition import PCA

pca = PCA(n_components=20)
principalComponents = pca.fit_transform(X)
principalDfX = pd.DataFrame(data = principalComponents)

In [9]:
principalDfX.head()
print(Counter(y))

Counter({6: 37000, 5: 18871, 3: 11132, 4: 6062, 2: 4089, 7: 3496, 0: 677, 1: 583, 8: 378, 9: 44})


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(principalDfX, y, test_size=0.2, random_state=42)

In [11]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train, y_train)


KNeighborsClassifier()

In [12]:
y_pred = neigh.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7426, 5: 3635, 3: 2454, 4: 1265, 2: 784, 7: 668, 0: 154, 1: 41, 8: 40})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [13]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   8    2   21   76   23    0    1    0    0    0]
 [   5    1    7   75   27    0    0    2    0    0]
 [  11   12  309  379   45    3    0   18    9    0]
 [  82   12  332 1560  192   14    2   74    7    0]
 [  46    8   44  175  878    3    9   44    5    0]
 [   0    1   15   68   15 3613    0   10    1    0]
 [   0    0    0    1    2    1 7414    0    0    0]
 [   2    4   55  105   67    0    0  489    1    0]
 [   0    1    1   10   15    1    0   30   17    0]
 [   0    0    0    5    1    0    0    1    0    0]]
Accuracy Score : 0.8677354709418837
Report : 
              precision    recall  f1-score   support

           0       0.05      0.06      0.06       131
           1       0.02      0.01      0.01       117
           2       0.39      0.39      0.39       786
           3       0.64      0.69      0.66      2275
           4       0.69      0.72      0.71      1212
           5       0.99      0.97      0.98      3723
           6       1.00  

In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8656342518788431
Report : 
              precision    recall  f1-score   support

           0       0.09      0.10      0.09       546
           1       0.01      0.00      0.00       466
           2       0.40      0.39      0.39      3303
           3       0.63      0.68      0.66      8857
           4       0.66      0.72      0.69      4850
           5       0.99      0.97      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.72      0.67      0.69      2773
           8       0.51      0.16      0.24       303
           9       0.00      0.00      0.00        37

    accuracy                           0.87     65865
   macro avg       0.50      0.47      0.47     65865
weighted avg       0.86      0.87      0.86     65865



In [15]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train, y_train)

SVC(decision_function_shape='ovo', gamma='auto')

In [16]:
y_pred=clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7435, 5: 3604, 3: 2146, 4: 1631, 2: 979, 7: 669, 8: 3})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [17]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   30   37   60    0    1    3    0    0]
 [   0    0    7   34   72    0    0    4    0    0]
 [   0    0  430  212  103    2    5   34    0    0]
 [   0    0  371 1558  266    7    7   65    1    0]
 [   0    0   56   99  987    0    4   65    1    0]
 [   0    0    6   85   27 3593    0   12    0    0]
 [   0    0    0    0    0    0 7418    0    0    0]
 [   0    0   72  106   94    2    0  449    0    0]
 [   0    0    7    9   21    0    0   37    1    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.876662415740572
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.55      0.49       786
           3       0.73      0.68      0.70      2275
           4       0.61      0.81      0.69      1212
           5       1.00      0.97      0.98      3723
           6       1.00   

In [18]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8750170803917103
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.44      0.50      0.47      3303
           3       0.71      0.68      0.69      8857
           4       0.61      0.82      0.70      4850
           5       1.00      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.65      0.65      0.65      2773
           8       0.44      0.01      0.03       303
           9       0.00      0.00      0.00        37

    accuracy                           0.88     65865
   macro avg       0.48      0.46      0.45     65865
weighted avg       0.87      0.88      0.87     65865



In [19]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train, y_train)

In [20]:
y_pred = clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7444, 5: 3998, 3: 2200, 4: 1361, 2: 877, 7: 583, 0: 4})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [21]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Accuracy Score : 0.8553470577518674
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.49      0.46       786
           3       0.68      0.65      0.67      2275
           4       0.60      0.67      0.63      1212
           5       0.90      0.97      0.93      3723
           6       1.00      1.00      1.00      7418
           7       0.67      0.54      0.59       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.86     16467
   macro avg       0.43      0.43      0.43     16467
weighted avg       0.84      0.86      0.85     16467



In [22]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8508312457299021
Report : 
              precision    recall  f1-score   support

           0       0.25      0.00      0.01       546
           1       0.00      0.00      0.00       466
           2       0.43      0.46      0.44      3303
           3       0.66      0.66      0.66      8857
           4       0.59      0.63      0.61      4850
           5       0.89      0.96      0.93     15148
           6       0.99      1.00      1.00     29582
           7       0.65      0.54      0.59      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.85     65865
   macro avg       0.45      0.42      0.42     65865
weighted avg       0.84      0.85      0.84     65865



In [23]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train, y_train)
clf

MLPClassifier(alpha=1)

In [24]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   25   23   30   45    5    3    0    0]
 [   0    0    4   24   41   45    0    3    0    0]
 [   1    0  384  250   61   56    5   29    0    0]
 [   3    0  348 1488  229  125   19   63    0    0]
 [   0    0   38  187  816  122    2   47    0    0]
 [   0    0    3   82   29 3597    1   11    0    0]
 [   0    0    0    5    1    0 7412    0    0    0]
 [   0    0   72  127  129    7    0  388    0    0]
 [   0    0    3    9   23    1    0   39    0    0]
 [   0    0    0    5    2    0    0    0    0    0]]
Accuracy Score : 0.8553470577518674
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.49      0.46       786
           3       0.68      0.65      0.67      2275
           4       0.60      0.67      0.63      1212
           5       0.90      0.97      0.93      3723
           6       1.00  

In [25]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8632657708950126
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.43      0.27      0.33      3303
           3       0.65      0.73      0.68      8857
           4       0.57      0.77      0.65      4850
           5       0.99      0.96      0.97     15148
           6       1.00      1.00      1.00     29582
           7       0.58      0.60      0.59      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.86     65865
   macro avg       0.42      0.43      0.42     65865
weighted avg       0.85      0.86      0.85     65865



In [26]:
#RandomForest
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train, y_train)
arr = forest.predict(X_train).astype(int)
print(classification_report(y_train, arr))

              precision    recall  f1-score   support

           0       0.87      0.79      0.83       546
           1       0.57      0.77      0.66       466
           2       0.42      0.89      0.57      3303
           3       0.66      0.54      0.59      8857
           4       0.76      0.52      0.62      4850
           5       0.92      0.96      0.94     15148
           6       0.94      0.97      0.96     29582
           7       0.76      0.22      0.34      2773
           8       1.00      0.01      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.83     65865
   macro avg       0.69      0.57      0.55     65865
weighted avg       0.85      0.83      0.82     65865



In [27]:
arr = forest.predict(X_test).astype(int)
print(classification_report(y_test, arr))

              precision    recall  f1-score   support

           0       0.09      0.03      0.05       131
           1       0.15      0.30      0.20       117
           2       0.28      0.61      0.38       786
           3       0.54      0.40      0.46      2275
           4       0.57      0.48      0.52      1212
           5       0.87      0.95      0.91      3723
           6       0.96      0.97      0.96      7418
           7       0.93      0.20      0.32       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.78     16467
   macro avg       0.44      0.39      0.38     16467
weighted avg       0.80      0.78      0.77     16467



In [28]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred  =  classifier.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.08      0.45      0.13       546
           1       0.01      0.03      0.02       466
           2       0.17      0.03      0.06      3303
           3       0.57      0.26      0.36      8857
           4       0.18      0.17      0.18      4850
           5       0.83      0.93      0.88     15148
           6       0.95      0.61      0.75     29582
           7       0.14      0.66      0.22      2773
           8       0.05      0.25      0.08       303
           9       0.01      0.11      0.01        37

    accuracy                           0.57     65865
   macro avg       0.30      0.35      0.27     65865
weighted avg       0.72      0.57      0.61     65865



In [29]:
y_pred  =  classifier.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.07      0.44      0.12       131
           1       0.02      0.04      0.02       117
           2       0.20      0.04      0.07       786
           3       0.60      0.26      0.37      2275
           4       0.20      0.18      0.19      1212
           5       0.83      0.93      0.88      3723
           6       0.94      0.62      0.75      7418
           7       0.13      0.63      0.22       723
           8       0.07      0.33      0.12        75
           9       0.00      0.00      0.00         7

    accuracy                           0.58     16467
   macro avg       0.31      0.35      0.27     16467
weighted avg       0.73      0.58      0.62     16467



In [30]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train, y_train)
y_pred = clf_entropy.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.37      0.72      0.49      8857
           4       0.00      0.00      0.00      4850
           5       0.83      0.92      0.87     15148
           6       0.91      0.98      0.94     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.75     65865
   macro avg       0.21      0.26      0.23     65865
weighted avg       0.65      0.75      0.69     65865



In [31]:
y_pred = clf_entropy.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.38      0.72      0.50      2275
           4       0.00      0.00      0.00      1212
           5       0.84      0.92      0.88      3723
           6       0.91      0.98      0.94      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.75     16467
   macro avg       0.21      0.26      0.23     16467
weighted avg       0.65      0.75      0.69     16467

